In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
print("Current working directory:")
print(os.getcwd())

print("\nFiles and folders:")
for item in os.listdir():
    print(item)

In [ ]:
import pandas as pd
import os

label_files = [
    "labels.csv",
    "labels_v2.csv",
    "labels_final.csv"
]

for file in label_files:
    print("=" * 60)
    print(f"FILE: {file}")
    
    if os.path.exists(file):
        df = pd.read_csv(file)
        
        print("Shape:", df.shape)
        print("Columns:", list(df.columns))
        print("\nFirst 5 rows:")
        display(df.head())
    else:
        print("❌ File not found")

In [ ]:
# Load final dataset

df = pd.read_csv("labels_final.csv")

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())

print("\nDuplicate rows:", df.duplicated().sum())

print("\nSBP statistics:")
print(df["SBP"].describe())

print("\nDBP statistics:")
print(df["DBP"].describe())

In [ ]:
# Extract patient ID from image filename

df["patient_id"] = df["image"].str.split("_").str[0]

print("Number of unique patients:", df["patient_id"].nunique())

print("\nPatients:")
print(df["patient_id"].unique()[:20])

print("\nSamples per patient:")
print(df["patient_id"].value_counts().head(20))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# SBP
axes[0].hist(df["SBP"], bins=40)
axes[0].set_title("SBP Distribution")
axes[0].set_xlabel("SBP (mmHg)")
axes[0].set_ylabel("Number of Samples")

# DBP
axes[1].hist(df["DBP"], bins=40)
axes[1].set_title("DBP Distribution")
axes[1].set_xlabel("DBP (mmHg)")
axes[1].set_ylabel("Number of Samples")

plt.tight_layout()
plt.show()

In [ ]:
# Patient-level BP statistics

patient_stats = df.groupby("patient_id").agg(
    samples=("image", "count"),
    sbp_min=("SBP", "min"),
    sbp_max=("SBP", "max"),
    sbp_mean=("SBP", "mean"),
    dbp_min=("DBP", "min"),
    dbp_max=("DBP", "max"),
    dbp_mean=("DBP", "mean")
)

print(patient_stats.to_string())

In [ ]:
# Find suspicious BP values

suspicious = df[
    (df["SBP"] < 70) |
    (df["SBP"] > 200) |
    (df["DBP"] < 40) |
    (df["DBP"] > 120)
]

print("Suspicious samples:", len(suspicious))

display(
    suspicious[
        ["image", "patient_id", "SBP", "DBP"]
    ].sort_values(["patient_id", "SBP"])
)

In [ ]:
import os

# Check possible image locations

possible_dirs = [
    "data",
    "data/images",
    "images",
    "scalograms"
]

for folder in possible_dirs:
    if os.path.exists(folder):
        print(f"✅ Found: {folder}")
        print("Number of files:", len(os.listdir(folder)))
    else:
        print(f"❌ Not found: {folder}")

In [ ]:
# Check image paths against the CSV

for folder in possible_dirs:
    if os.path.exists(folder):
        df["test_path"] = df["image"].apply(
            lambda x: os.path.join(folder, x)
        )
        
        exists = df["test_path"].apply(os.path.exists)
        
        print("\nFolder:", folder)
        print("Images found:", exists.sum())
        print("Images missing:", (~exists).sum())

In [ ]:
# Number of unique SBP/DBP values per patient

patient_label_check = df.groupby("patient_id").agg(
    unique_sbp=("SBP", "nunique"),
    unique_dbp=("DBP", "nunique"),
    samples=("image", "count")
)

display(patient_label_check.sort_values("unique_sbp", ascending=False).head(20))

In [ ]:
import os

print("Searching for PNG images...\n")

png_files = []

for root, dirs, files in os.walk("."):
    for file in files:
        if file.lower().endswith(".png"):
            png_files.append(os.path.join(root, file))

print("Total PNG files found:", len(png_files))

print("\nFirst 30 PNG files:")
for path in png_files[:30]:
    print(path)

In [ ]:
target = "p10020306_83404654_83404654_0005_0.png"

matches = []

for root, dirs, files in os.walk("."):
    if target in files:
        matches.append(os.path.join(root, target))

print("Matches found:", len(matches))

for path in matches:
    print(path)

In [ ]:
print("DATA FOLDER CONTENTS:\n")

for item in os.listdir("data"):
    full_path = os.path.join("data", item)

    if os.path.isdir(full_path):
        print("📁", item)
    else:
        print("📄", item)

In [ ]:
# Search project Python files for code related to SBP/DBP label generation

keywords = ["SBP", "DBP", "labels_final", "scalogram"]

for root, dirs, files in os.walk("."):
    for file in files:
        if file.endswith(".py") or file.endswith(".ipynb"):
            path = os.path.join(root, file)

            try:
                with open(path, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()

                if any(keyword in text for keyword in keywords):
                    print("\nFOUND:", path)

            except:
                pass

In [ ]:
from pathlib import Path
import os
import pandas as pd
import numpy as np
import wfdb

# --------------------------------------------------
# Select suspicious sample
# --------------------------------------------------

sample = df.iloc[0]

print("Image:", sample["image"])
print("Patient:", sample["patient_id"])
print("SBP:", sample["SBP"])
print("DBP:", sample["DBP"])

In [ ]:
filename = sample["image"].replace(".png", "")

parts = filename.split("_")

patient = parts[0]
record = parts[1]
segment = parts[2] + "_" + parts[3]
window_id = parts[4]

print("Patient :", patient)
print("Record  :", record)
print("Segment :", segment)
print("Window  :", window_id)

In [ ]:
from pathlib import Path

MIMIC_ROOT = Path(
    r"C:\Users\diksh\OneDrive\Documents\MIMIC4\MIMIC4\physionet.org\files\mimic4wdb\0.1.0"
)

record_path = (
    MIMIC_ROOT
    / "waves"
    / patient[:4]
    / patient
    / record
    / segment
)

print("Original record:")
print(record_path)

print("\nExists:", record_path.exists())

In [ ]:
rec = wfdb.rdrecord(str(record_path))

print("Signal names:")
print(rec.sig_name)

print("\nSampling frequency:")
print(rec.fs)

print("\nSignal shape:")
print(rec.p_signal.shape)

In [ ]:
channels = [c.upper() for c in rec.sig_name]

print("Channels:")
print(channels)

In [ ]:
abp_index = channels.index("ABP")

abp = rec.p_signal[:, abp_index]

abp = abp[np.isfinite(abp)]

print("Original ABP statistics:")
print("Min :", np.min(abp))
print("Max :", np.max(abp))
print("Mean:", np.mean(abp))
print("95th percentile:", np.percentile(abp, 95))
print("5th percentile :", np.percentile(abp, 5))

In [ ]:
abp_filtered = abp[
    (abp > 20) &
    (abp < 250)
]

print("\nAfter filtering:")
print("Min :", np.min(abp_filtered))
print("Max :", np.max(abp_filtered))
print("95th percentile:", np.percentile(abp_filtered, 95))
print("5th percentile :", np.percentile(abp_filtered, 5))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

plt.figure(figsize=(15, 5))

plt.plot(abp, linewidth=0.8)

plt.title("Original ABP Signal")
plt.xlabel("Sample")
plt.ylabel("ABP (mmHg)")
plt.grid(True)

plt.show()

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(abp_filtered, linewidth=0.8)

plt.title("Filtered ABP Signal: 20–250 mmHg")
plt.xlabel("Sample")
plt.ylabel("ABP (mmHg)")
plt.grid(True)

plt.show()

In [ ]:
samples_30s = int(rec.fs * 30)

plt.figure(figsize=(15, 5))

plt.plot(
    abp[:samples_30s],
    linewidth=1
)

plt.title("ABP — First 30 Seconds")
plt.xlabel("Sample")
plt.ylabel("ABP (mmHg)")
plt.grid(True)

plt.show()

In [ ]:
pleth_index = channels.index("PLETH")

ppg = rec.p_signal[:, pleth_index]

print("PPG statistics")
print("Min :", np.min(ppg))
print("Max :", np.max(ppg))
print("Mean:", np.mean(ppg))
print("Std :", np.std(ppg))

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    ppg[:samples_30s],
    linewidth=1
)

plt.title("PPG / PLETH — First 30 Seconds")
plt.xlabel("Sample")
plt.ylabel("PLETH")
plt.grid(True)

plt.show()

In [ ]:
WINDOW_SECONDS = 10
WINDOW = int(WINDOW_SECONDS * rec.fs)

print("Sampling frequency:", rec.fs)
print("Samples per window:", WINDOW)
print("Number of possible windows:", len(ppg) // WINDOW)

In [ ]:
for i in range(6):

    start = i * WINDOW
    end = start + WINDOW

    abp_window = abp[start:end]
    ppg_window = ppg[start:end]

    print("=" * 60)
    print("WINDOW:", i)

    print(
        "ABP min:",
        np.min(abp_window),
        "max:",
        np.max(abp_window),
        "mean:",
        np.mean(abp_window),
        "std:",
        np.std(abp_window)
    )

    print(
        "PPG min:",
        np.min(ppg_window),
        "max:",
        np.max(ppg_window),
        "mean:",
        np.mean(ppg_window),
        "std:",
        np.std(ppg_window)
    )

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(15, 14))

for i in range(6):

    start = i * WINDOW
    end = start + WINDOW

    axes[i].plot(abp[start:end])

    axes[i].set_title(f"ABP Window {i}")
    axes[i].set_ylabel("mmHg")
    axes[i].grid(True)

axes[-1].set_xlabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(15, 14))

for i in range(6):

    start = i * WINDOW
    end = start + WINDOW

    axes[i].plot(ppg[start:end])

    axes[i].set_title(f"PPG Window {i}")
    axes[i].set_ylabel("PLETH")
    axes[i].grid(True)

axes[-1].set_xlabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# Scan ALL 10-second windows
# ==========================================

window_results = []

for i in range(len(abp) // WINDOW):

    start = i * WINDOW
    end = start + WINDOW

    abp_window = abp[start:end]

    if len(abp_window) < WINDOW:
        continue

    p5 = np.percentile(abp_window, 5)
    p95 = np.percentile(abp_window, 95)

    window_results.append({
        "window": i,
        "start_sample": start,
        "end_sample": end,
        "p5": p5,
        "p95": p95,
        "mean": np.mean(abp_window),
        "std": np.std(abp_window),
        "min": np.min(abp_window),
        "max": np.max(abp_window),
        "range": np.max(abp_window) - np.min(abp_window),
        "pulse_pressure": p95 - p5
    })

window_df = pd.DataFrame(window_results)

print("Total windows analyzed:", len(window_df))

display(window_df.head(20))

In [ ]:
# ==========================================
# Initial ABP quality screening
# ==========================================

valid_windows = window_df[
    (window_df["p95"] >= 80) &
    (window_df["p95"] <= 200) &
    (window_df["p5"] >= 40) &
    (window_df["p5"] <= 120) &
    (window_df["pulse_pressure"] >= 20)
].copy()

print("Potentially valid windows:", len(valid_windows))

display(valid_windows)

In [ ]:
plt.figure(figsize=(15, 5))

plt.plot(
    window_df["window"],
    window_df["p95"],
    label="ABP P95"
)

plt.plot(
    window_df["window"],
    window_df["p5"],
    label="ABP P5"
)

plt.axhline(80, linestyle="--")
plt.axhline(120, linestyle="--")

plt.xlabel("10-second Window Number")
plt.ylabel("ABP (mmHg)")
plt.title("ABP Percentiles Across Entire Record")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# Sort by pulse pressure to find windows
# with the strongest ABP waveform

best_windows = window_df.sort_values(
    "pulse_pressure",
    ascending=False
).head(20)

display(best_windows)

In [ ]:
top_windows = best_windows["window"].head(6).tolist()

fig, axes = plt.subplots(
    len(top_windows),
    1,
    figsize=(15, 14)
)

for ax, i in zip(axes, top_windows):

    start = i * WINDOW
    end = start + WINDOW

    ax.plot(abp[start:end])

    ax.set_title(f"ABP Window {i}")
    ax.set_ylabel("mmHg")
    ax.grid(True)

axes[-1].set_xlabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
# ==========================================
# STRONGER ABP QUALITY FILTER
# ==========================================

quality_df = window_df.copy()

valid_fractions = []

for i in quality_df["window"]:

    start = i * WINDOW
    end = start + WINDOW

    w = abp[start:end]

    # Fraction of samples inside reasonable ABP range
    valid_fraction = np.mean(
        (w >= 40) & (w <= 180)
    )

    valid_fractions.append(valid_fraction)


# Add the calculated values
quality_df["valid_fraction"] = valid_fractions


# ------------------------------------------
# Quality filtering
# ------------------------------------------

good_windows = quality_df[
    (quality_df["p5"] >= 45) &
    (quality_df["p95"] <= 180) &
    (quality_df["pulse_pressure"] >= 20) &
    (quality_df["pulse_pressure"] <= 80) &
    (quality_df["valid_fraction"] >= 0.90) &
    (quality_df["min"] >= 30) &
    (quality_df["max"] <= 200)
].copy()


print("Total windows:", len(quality_df))
print("Good windows:", len(good_windows))
print("Rejected windows:", len(quality_df) - len(good_windows))

display(
    good_windows[
        [
            "window",
            "p5",
            "p95",
            "mean",
            "std",
            "min",
            "max",
            "pulse_pressure",
            "valid_fraction"
        ]
    ]
)

In [ ]:
good_windows["SBP_est"] = good_windows["p95"]
good_windows["DBP_est"] = good_windows["p5"]

display(
    good_windows[
        [
            "window",
            "SBP_est",
            "DBP_est",
            "pulse_pressure"
        ]
    ]
)

In [ ]:
accepted = good_windows["window"].astype(int).tolist()

print("Accepted windows:")
print(accepted)

fig, axes = plt.subplots(
    len(accepted),
    1,
    figsize=(15, max(4, len(accepted) * 3))
)

if len(accepted) == 1:
    axes = [axes]

for ax, i in zip(axes, accepted):

    start = i * WINDOW
    end = start + WINDOW

    ax.plot(abp[start:end])

    ax.set_title(
        f"ABP Window {i} | "
        f"SBP≈{window_df.loc[window_df.window==i, 'p95'].iloc[0]:.1f} | "
        f"DBP≈{window_df.loc[window_df.window==i, 'p5'].iloc[0]:.1f}"
    )

    ax.set_ylabel("mmHg")
    ax.grid(True)

axes[-1].set_xlabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(
    len(accepted),
    1,
    figsize=(15, max(4, len(accepted) * 3))
)

if len(accepted) == 1:
    axes = [axes]

for ax, i in zip(axes, accepted):

    start = i * WINDOW
    end = start + WINDOW

    ax.plot(ppg[start:end])

    ax.set_title(f"PPG Window {i}")
    ax.set_ylabel("PLETH")
    ax.grid(True)

axes[-1].set_xlabel("Sample")

plt.tight_layout()
plt.show()

In [ ]:
from scipy.signal import find_peaks
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ==========================================
# BEAT-LEVEL ABP PEAK / TROUGH DETECTION
# ==========================================

beat_results = []

for i in good_windows["window"].astype(int):

    start = i * WINDOW
    end = start + WINDOW

    abp_window = abp[start:end]

    # --------------------------------------
    # Detect systolic peaks
    # --------------------------------------
    peaks, peak_props = find_peaks(
        abp_window,
        distance=int(rec.fs * 0.45),
        prominence=5
    )

    # --------------------------------------
    # Detect diastolic troughs
    # --------------------------------------
    troughs, trough_props = find_peaks(
        -abp_window,
        distance=int(rec.fs * 0.45),
        prominence=3
    )

    peak_values = abp_window[peaks]
    trough_values = abp_window[troughs]

    # Keep physiologically reasonable values
    peak_values = peak_values[
        (peak_values >= 70) &
        (peak_values <= 180)
    ]

    trough_values = trough_values[
        (trough_values >= 40) &
        (trough_values <= 120)
    ]

    beat_results.append({
        "window": i,
        "num_systolic_peaks": len(peak_values),
        "num_diastolic_troughs": len(trough_values),
        "SBP_median": np.median(peak_values) if len(peak_values) > 0 else np.nan,
        "DBP_median": np.median(trough_values) if len(trough_values) > 0 else np.nan,
        "SBP_mean": np.mean(peak_values) if len(peak_values) > 0 else np.nan,
        "DBP_mean": np.mean(trough_values) if len(trough_values) > 0 else np.nan
    })

beat_results_df = pd.DataFrame(beat_results)

display(beat_results_df)

In [ ]:
# ==========================================
# VISUALIZE DETECTED ABP BEATS
# ==========================================

for i in good_windows["window"].astype(int):

    start = i * WINDOW
    end = start + WINDOW

    abp_window = abp[start:end]

    peaks, _ = find_peaks(
        abp_window,
        distance=int(rec.fs * 0.45),
        prominence=5
    )

    troughs, _ = find_peaks(
        -abp_window,
        distance=int(rec.fs * 0.45),
        prominence=3
    )

    plt.figure(figsize=(15, 5))

    plt.plot(abp_window, label="ABP")

    plt.scatter(
        peaks,
        abp_window[peaks],
        marker="o",
        label="Systolic peaks"
    )

    plt.scatter(
        troughs,
        abp_window[troughs],
        marker="x",
        label="Diastolic troughs"
    )

    plt.title(f"ABP Beat Detection — Window {i}")
    plt.xlabel("Sample")
    plt.ylabel("ABP (mmHg)")
    plt.grid(True)
    plt.legend()

    plt.show()

In [ ]:
comparison = good_windows[
    ["window", "SBP_est", "DBP_est"]
].merge(
    beat_results_df[
        [
            "window",
            "num_systolic_peaks",
            "num_diastolic_troughs",
            "SBP_median",
            "DBP_median",
            "SBP_mean",
            "DBP_mean"
        ]
    ],
    on="window"
)

display(comparison)

In [ ]:
# ==========================================
# VALIDATED WINDOW-LEVEL BP LABELS
# ==========================================

validated_labels = beat_results_df.copy()

validated_labels = validated_labels[
    [
        "window",
        "num_systolic_peaks",
        "num_diastolic_troughs",
        "SBP_median",
        "DBP_median",
        "SBP_mean",
        "DBP_mean"
    ]
].copy()

# Keep only windows with enough detected beats
validated_labels = validated_labels[
    (validated_labels["num_systolic_peaks"] >= 8) &
    (validated_labels["num_diastolic_troughs"] >= 8) &
    (validated_labels["SBP_median"].notna()) &
    (validated_labels["DBP_median"].notna())
].copy()

print("Validated windows:", len(validated_labels))

display(validated_labels)

In [ ]:
# ==========================================
# ADD PATIENT / RECORD INFORMATION
# ==========================================

validated_labels["patient"] = patient
validated_labels["record"] = record
validated_labels["fs"] = rec.fs

validated_labels["start_sample"] = (
    validated_labels["window"] * WINDOW
)

validated_labels["end_sample"] = (
    validated_labels["start_sample"] + WINDOW
)

validated_labels["duration_seconds"] = (
    WINDOW / rec.fs
)

validated_labels = validated_labels[
    [
        "patient",
        "record",
        "window",
        "start_sample",
        "end_sample",
        "duration_seconds",
        "num_systolic_peaks",
        "num_diastolic_troughs",
        "SBP_median",
        "DBP_median",
        "SBP_mean",
        "DBP_mean"
    ]
]

display(validated_labels)

In [ ]:
validated_labels.to_csv(
    "validated_window_labels_test.csv",
    index=False
)

print("Saved:")
print("validated_window_labels_test.csv")

In [ ]:
import os
import glob

# ==========================================
# INSPECT SCALOGRAM NAMES FOR THIS RECORD
# ==========================================

record_prefix = f"{patient}_{record}"

matches = []

for path in glob.glob(
    os.path.join("scalograms_final", "*.png")
):
    
    filename = os.path.basename(path)
    
    if filename.startswith(record_prefix):
        matches.append(filename)

print("Total scalograms for this record:", len(matches))

print("\nFirst 30:")
for x in matches[:30]:
    print(x)

In [ ]:
# ==========================================
# FIND SCALOGRAMS FOR WINDOWS 311–313
# ==========================================

for w in [311, 312, 313]:

    print("\n" + "=" * 60)
    print("WINDOW:", w)

    token = f"_{w}_"

    window_matches = [
        x for x in matches
        if token in x
    ]

    print("Number of images:", len(window_matches))

    for x in window_matches:
        print(x)

In [ ]:
import os

TEST_OUTPUT_DIR = "scalograms_validated_test"
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

In [ ]:
# ==========================================
# GENERATE VALIDATED SCALOGRAMS
# FOR WINDOWS 311, 312, 313
# ==========================================

import os
import cv2
import pywt
import numpy as np
import pandas as pd

TEST_OUTPUT_DIR = "scalograms_validated_test"
os.makedirs(TEST_OUTPUT_DIR, exist_ok=True)

accepted_windows = [311, 312, 313]

validated_scalogram_labels = []

for w in accepted_windows:

    start = w * WINDOW
    end = start + WINDOW

    # PPG corresponding to THIS 10-sec window
    ppg_window = ppg[start:end]

    if len(ppg_window) != WINDOW:
        print("Skipping window:", w)
        continue

    # --------------------------------------
    # CWT
    # --------------------------------------

    coef, _ = pywt.cwt(
        ppg_window,
        np.arange(1, 129),
        "morl"
    )

    img = np.abs(coef)

    # Normalize
    img -= img.min()

    if img.max() > 0:
        img /= img.max()

    img = (img * 255).astype(np.uint8)

    # Convert to RGB heatmap
    img = cv2.applyColorMap(
        img,
        cv2.COLORMAP_JET
    )

    # Resize for Transformer/CNN
    img = cv2.resize(
        img,
        (224, 224),
        interpolation=cv2.INTER_AREA
    )

    # --------------------------------------
    # Filename
    # --------------------------------------

    filename = (
        f"{patient}_{record}_{segment}_window_{w}.png"
    )

    save_path = os.path.join(
        TEST_OUTPUT_DIR,
        filename
    )

    cv2.imwrite(save_path, img)

    # --------------------------------------
    # Get beat-level BP labels
    # --------------------------------------

    row = beat_results_df[
        beat_results_df["window"] == w
    ].iloc[0]

    validated_scalogram_labels.append({
        "image": filename,
        "patient": patient,
        "record": record,
        "segment": segment,
        "window": w,
        "SBP": row["SBP_median"],
        "DBP": row["DBP_median"]
    })

    print(
        f"Saved window {w}: "
        f"SBP={row['SBP_median']:.2f}, "
        f"DBP={row['DBP_median']:.2f}"
    )


validated_test_df = pd.DataFrame(
    validated_scalogram_labels
)

display(validated_test_df)

In [ ]:
validated_test_df.to_csv(
    "validated_scalogram_labels_test.csv",
    index=False
)

print("Saved:")
print("validated_scalogram_labels_test.csv")

In [ ]:
import glob

files = glob.glob(
    os.path.join(
        TEST_OUTPUT_DIR,
        "*.png"
    )
)

print("Images generated:", len(files))

for f in files:
    print(f)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(
    3,
    1,
    figsize=(10, 12)
)

for ax, (_, row) in zip(
    axes,
    validated_test_df.iterrows()
):

    image_path = os.path.join(
        TEST_OUTPUT_DIR,
        row["image"]
    )

    img = cv2.imread(image_path)
    img = cv2.cvtColor(
        img,
        cv2.COLOR_BGR2RGB
    )

    ax.imshow(img)

    ax.set_title(
        f"Window {row['window']} | "
        f"SBP={row['SBP']:.2f} | "
        f"DBP={row['DBP']:.2f}"
    )

    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
valid_segments = good_windows.copy()

print("Valid segments:", len(valid_segments))
display(valid_segments)

In [ ]:
print(good_windows.columns.tolist())

In [ ]:
# ============================================================
# PREPARE ACCEPTED WINDOWS FOR FINAL SCALOGRAM GENERATION
# ============================================================

valid_segments = good_windows.copy()

# Metadata of the record we validated
valid_segments["patient"] = "p10020306"
valid_segments["record"] = "83404654"
valid_segments["segment"] = "83404654_0005"

# Use our validated window estimates as BP labels
valid_segments["SBP"] = valid_segments["SBP_est"]
valid_segments["DBP"] = valid_segments["DBP_est"]

print("Valid windows:", len(valid_segments))

display(
    valid_segments[
        [
            "patient",
            "record",
            "segment",
            "window",
            "start_sample",
            "end_sample",
            "SBP",
            "DBP"
        ]
    ]
)

In [ ]:
from pathlib import Path

WAVE_ROOT = Path(
    r"C:\Users\diksh\OneDrive\Documents\MIMIC4\MIMIC4\physionet.org\files\mimic4wdb\0.1.0"
)

print("WAVE_ROOT exists:", WAVE_ROOT.exists())

In [ ]:
record_path = (
    WAVE_ROOT /
    "waves" /
    "p100" /
    "p10020306" /
    "83404654" /
    "83404654_0005"
)

print("Record:", record_path)
print("Header exists:", Path(str(record_path) + ".hea").exists())
print("Files in folder:")

for f in record_path.parent.glob("83404654_0005*"):
    print(" ", f.name)

In [ ]:
import numpy as np
from scipy.signal import butter, filtfilt


def preprocess_ppg(ppg, fs):
    """
    Basic PPG preprocessing:
    - Remove NaN/Inf
    - Remove extreme invalid values
    - Bandpass filter
    - Normalize signal
    """

    ppg = np.asarray(ppg, dtype=np.float64)

    # Remove NaN / Inf
    ppg = np.nan_to_num(
        ppg,
        nan=np.nanmedian(ppg),
        posinf=np.nanmedian(ppg),
        neginf=np.nanmedian(ppg)
    )

    # Remove DC component
    ppg = ppg - np.median(ppg)

    # Bandpass filter
    lowcut = 0.5
    highcut = min(8.0, fs / 2 - 0.5)

    if highcut <= lowcut:
        return ppg

    b, a = butter(
        3,
        [lowcut / (fs / 2),
         highcut / (fs / 2)],
        btype="band"
    )

    filtered = filtfilt(b, a, ppg)

    # Normalize
    std = np.std(filtered)

    if std > 0:
        filtered = (filtered - np.mean(filtered)) / std

    return filtered

In [ ]:
print("preprocess_ppg exists:", callable(preprocess_ppg))

In [ ]:
import os
import cv2
import pywt
import wfdb
import numpy as np
import pandas as pd
from tqdm import tqdm

OUTPUT_DIR = "scalograms_final"
os.makedirs(OUTPUT_DIR, exist_ok=True)

WINDOW_SECONDS = 10

all_labels = []
saved = 0

for _, row in tqdm(
    valid_segments.iterrows(),
    total=len(valid_segments),
    desc="Generating final scalograms"
):

    patient = row["patient"]
    record = row["record"]
    segment = row["segment"]
    window_id = int(row["window"])

    rrecord_path = (
        WAVE_ROOT /
        "waves" /
        patient[:4] /
        patient /
        record /
        segment
    )

    try:
        rec = wfdb.rdrecord(str(record_path))
    except Exception as e:
        print("Could not read:", record_path)
        print(e)
        continue

    channels = [c.upper() for c in rec.sig_name]

    if "PLETH" in channels:
        idx = channels.index("PLETH")
    elif "PPG" in channels:
        idx = channels.index("PPG")
    else:
        print("No PLETH/PPG:", segment)
        continue

    ppg = rec.p_signal[:, idx]

    try:
        processed = preprocess_ppg(ppg, rec.fs)
    except Exception as e:
        print("Preprocessing failed:", e)
        continue

    WINDOW = int(WINDOW_SECONDS * rec.fs)

    start = window_id * WINDOW
    end = start + WINDOW

    window = processed[start:end]

    if len(window) != WINDOW:
        print("Invalid window:", window_id)
        continue

    # --------------------------------------------------------
    # CWT
    # --------------------------------------------------------

    coef, _ = pywt.cwt(
        window,
        np.arange(1, 129),
        "morl"
    )

    img = np.abs(coef)

    img -= img.min()

    if img.max() > 0:
        img /= img.max()

    img = (img * 255).astype(np.uint8)

    img = cv2.applyColorMap(
        img,
        cv2.COLORMAP_JET
    )

    img = cv2.resize(
        img,
        (224, 224),
        interpolation=cv2.INTER_AREA
    )

    # --------------------------------------------------------
    # SAVE
    # --------------------------------------------------------

    filename = (
        f"{patient}_{record}_{segment}_{window_id}.png"
    )

    save_path = os.path.join(
        OUTPUT_DIR,
        filename
    )

    success = cv2.imwrite(save_path, img)

    if not success:
        print("Failed to save:", save_path)
        continue

    # --------------------------------------------------------
    # LABEL
    # --------------------------------------------------------

    all_labels.append({
        "image": filename,
        "patient": patient,
        "record": record,
        "segment": segment,
        "window": window_id,
        "SBP": row["SBP"],
        "DBP": row["DBP"]
    })

    saved += 1


# ============================================================
# SAVE LABEL CSV
# ============================================================

labels_final = pd.DataFrame(all_labels)

labels_final.to_csv(
    "labels_final.csv",
    index=False
)

print("=" * 60)
print("FINAL DATASET GENERATED")
print("=" * 60)
print("Images Generated :", saved)
print("Labels Saved     :", len(labels_final))
print("=" * 60)

display(labels_final)

In [ ]:
# ============================================================
# CHECK WHY WINDOWS ARE BEING REJECTED
# ============================================================

print("Total windows:", len(window_df))

# Show statistics for all windows
display(
    window_df[
        [
            "window",
            "p5",
            "p95",
            "mean",
            "std",
            "min",
            "max",
            "range",
            "pulse_pressure"
        ]
    ].describe()
)

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(12, 5))

plt.plot(
    window_df["window"],
    window_df["p95"],
    label="P95"
)

plt.plot(
    window_df["window"],
    window_df["p5"],
    label="P5"
)

plt.axhline(120, linestyle="--", label="120 mmHg")
plt.axhline(80, linestyle="--", label="80 mmHg")

plt.xlabel("Window")
plt.ylabel("ABP (mmHg)")
plt.title("ABP P5/P95 Across Windows")
plt.legend()
plt.grid(True)

plt.show()

In [ ]:
# ============================================================
# FIND HOW MANY SEGMENTS HAVE USABLE ABP
# ============================================================

print("Total valid segments available:", len(valid_segments))

display(
    valid_segments[
        ["patient", "record", "segment", "SBP", "DBP"]
    ].head(20)
)

In [ ]:
# ============================================================
# CHECK SEGMENT DISTRIBUTION
# ============================================================

print("Unique patients :", valid_segments["patient"].nunique())
print("Unique records  :", valid_segments["record"].nunique())
print("Total segments  :", len(valid_segments))

print("\nSegments per patient:")
display(
    valid_segments.groupby("patient").size()
    .sort_values(ascending=False)
    .head(20)
)

In [ ]:
# Show all DataFrame variables currently available

for name, obj in list(globals().items()):
    if isinstance(obj, pd.DataFrame):
        print(
            name,
            "→",
            obj.shape,
            "→",
            list(obj.columns)
        )

In [ ]:
# ============================================================
# STEP: REBUILD ALL UNIQUE RECORD SEGMENTS FROM ORIGINAL DATA
# ============================================================

import pandas as pd

# Start from the ORIGINAL dataset
source_df = df.copy()

print("Original samples:", len(source_df))
print("Original patients:", source_df["patient_id"].nunique())

# ------------------------------------------------------------
# Extract patient, record, segment and window from image name
# Example:
# p10020306_83404654_83404654_0005_0.png
# ------------------------------------------------------------

def parse_image_name(filename):
    name = str(filename).replace(".png", "")
    parts = name.split("_")

    if len(parts) < 5:
        return pd.Series({
            "patient": None,
            "record": None,
            "segment": None,
            "window": None
        })

    return pd.Series({
        "patient": parts[0],
        "record": parts[1],
        "segment": parts[2] + "_" + parts[3],
        "window": int(parts[4])
    })


parsed = source_df["image"].apply(parse_image_name)

source_df = pd.concat(
    [source_df, parsed],
    axis=1
)

# Remove rows that could not be parsed
source_df = source_df.dropna(
    subset=["patient", "record", "segment"]
)

print("\nParsed samples:", len(source_df))
print("Unique patients:", source_df["patient"].nunique())
print("Unique records:", source_df["record"].nunique())
print("Unique segments:", source_df["segment"].nunique())

# ------------------------------------------------------------
# Create ONE row per waveform segment
# ------------------------------------------------------------

all_segments = (
    source_df[
        [
            "patient",
            "record",
            "segment",
            "SBP",
            "DBP"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nTotal unique segments:", len(all_segments))

display(all_segments.head(20))

In [ ]:
# ============================================================
# SCAN ALL 610 SEGMENTS FOR ABP WINDOWS
# ============================================================

import numpy as np
import pandas as pd
import wfdb
from tqdm import tqdm

WINDOW_SECONDS = 10

all_window_results = []

for _, row in tqdm(
    all_segments.iterrows(),
    total=len(all_segments),
    desc="Scanning all segments"
):

    patient = str(row["patient"])
    record = str(row["record"])
    segment = str(row["segment"])

    # WFDB RECORD BASE PATH
    record_path = (
        WAVE_ROOT /
        "waves" /
        patient[:4] /
        patient /
        record /
        segment
    )

    # Check the actual WFDB header file
    header_path = record_path.with_suffix(".hea")

    if not header_path.exists():
        continue

    try:
        rec = wfdb.rdrecord(str(record_path))
    except Exception as e:
        continue

    channels = [str(c).upper() for c in rec.sig_name]

    # --------------------------------------------------------
    # Find ABP channel
    # --------------------------------------------------------

    if "ABP" not in channels:
        continue

    abp_index = channels.index("ABP")

    abp = rec.p_signal[:, abp_index]

    # --------------------------------------------------------
    # Sampling frequency
    # --------------------------------------------------------

    fs = float(rec.fs)

    WINDOW = int(WINDOW_SECONDS * fs)

    if len(abp) < WINDOW:
        continue

    num_windows = len(abp) // WINDOW

    # --------------------------------------------------------
    # Process every 10-second window
    # --------------------------------------------------------

    for window_id in range(num_windows):

        start = window_id * WINDOW
        end = start + WINDOW

        window = abp[start:end]

        if len(window) != WINDOW:
            continue

        finite_mask = np.isfinite(window)

        if finite_mask.sum() < 0.95 * len(window):
            continue

        clean_window = window[finite_mask]

        p5 = np.percentile(clean_window, 5)
        p95 = np.percentile(clean_window, 95)

        mean = np.mean(clean_window)
        std = np.std(clean_window)

        minimum = np.min(clean_window)
        maximum = np.max(clean_window)

        value_range = maximum - minimum
        pulse_pressure = p95 - p5

        all_window_results.append({
            "patient": patient,
            "record": record,
            "segment": segment,
            "window": window_id,

            "start_sample": start,
            "end_sample": end,

            "p5": p5,
            "p95": p95,
            "mean": mean,
            "std": std,

            "min": minimum,
            "max": maximum,
            "range": value_range,
            "pulse_pressure": pulse_pressure,

            "fs": fs,

            "SBP_label": row["SBP"],
            "DBP_label": row["DBP"]
        })


# ------------------------------------------------------------
# Create dataframe
# ------------------------------------------------------------

all_windows_df = pd.DataFrame(all_window_results)

print("=" * 60)
print("ALL SEGMENTS SCANNED")
print("=" * 60)

print("Source segments :", len(all_segments))
print("Source patients :", all_segments["patient"].nunique())
print("Windows found   :", len(all_windows_df))
print("=" * 60)

if len(all_windows_df) > 0:

    print(
        "Unique patients:",
        all_windows_df["patient"].nunique()
    )

    print(
        "Unique segments:",
        all_windows_df["segment"].nunique()
    )

    display(
        all_windows_df.head(20)
    )

else:

    print("NO WINDOWS FOUND")

In [ ]:
print("Total windows:", len(all_windows_df))
print("Unique patients:", all_windows_df["patient"].nunique())
print("Unique segments:", all_windows_df["segment"].nunique())

In [ ]:
# ============================================================
# DIAGNOSE WAVEFORM PATH
# ============================================================

print("WAVE_ROOT:")
print(WAVE_ROOT)

print("\nFirst segment:")
display(all_segments.head(1))

row = all_segments.iloc[0]

patient = row["patient"]
record = row["record"]
segment = row["segment"]

print("\nPatient :", patient)
print("Record  :", record)
print("Segment :", segment)

test_path = (
    WAVE_ROOT /
    "waves" /
    patient[:4] /
    patient /
    str(record) /
    str(segment)
)

print("\nConstructed path:")
print(test_path)

print("\nExists:", test_path.exists())

# Check parent folders too
print("\nParent exists:", test_path.parent.exists())
print("Record folder exists:", (test_path.parent).exists())

In [ ]:
# ============================================================
# INSPECT ALL WINDOWS
# ============================================================

print("Total windows:", len(all_windows_df))

display(
    all_windows_df[
        [
            "p5",
            "p95",
            "mean",
            "std",
            "min",
            "max",
            "range",
            "pulse_pressure"
        ]
    ].describe()
)

In [ ]:
# ============================================================
# CHECK LABEL DISTRIBUTION
# ============================================================

print("SBP range:")
print(
    all_windows_df["SBP_label"].describe()
)

print("\nDBP range:")
print(
    all_windows_df["DBP_label"].describe()
)

print("\nUnique patients:")
print(
    all_windows_df["patient"].nunique()
)

In [ ]:
# ============================================================
# QUALITY FILTERING — ALL WINDOWS
# ============================================================

quality_all = all_windows_df.copy()

# Fraction of samples inside a physiologically reasonable ABP range
quality_all["valid_fraction"] = (
    (quality_all["min"] >= 30) &
    (quality_all["max"] <= 250)
).astype(float)

print("Total windows:", len(quality_all))

# ------------------------------------------------------------
# FILTER 1: Remove extreme signal artifacts
# ------------------------------------------------------------

filter_artifact = (
    (quality_all["min"] >= 30) &
    (quality_all["max"] <= 250)
)

# ------------------------------------------------------------
# FILTER 2: Reasonable pulse pressure
# ------------------------------------------------------------

filter_pulse = (
    (quality_all["pulse_pressure"] >= 20) &
    (quality_all["pulse_pressure"] <= 100)
)

# ------------------------------------------------------------
# FILTER 3: Reasonable waveform variability
# ------------------------------------------------------------

filter_std = (
    (quality_all["std"] >= 1) &
    (quality_all["std"] <= 40)
)

# ------------------------------------------------------------
# COMBINE FILTERS
# ------------------------------------------------------------

good_windows_all = quality_all[
    filter_artifact &
    filter_pulse &
    filter_std
].copy()

print("=" * 60)
print("QUALITY FILTER RESULTS")
print("=" * 60)

print("Original windows :", len(quality_all))
print("Good windows     :", len(good_windows_all))
print("Rejected windows :", len(quality_all) - len(good_windows_all))

print(
    "Retention        :",
    round(len(good_windows_all) / len(quality_all) * 100, 2),
    "%"
)

display(
    good_windows_all[
        [
            "window",
            "p5",
            "p95",
            "mean",
            "std",
            "min",
            "max",
            "range",
            "pulse_pressure",
            "SBP_label",
            "DBP_label"
        ]
    ].head(20)
)

In [ ]:
# ============================================================
# PATIENT DISTRIBUTION AFTER QUALITY FILTERING
# ============================================================

print("Unique patients:", good_windows_all["patient"].nunique())
print("Unique segments:", good_windows_all["segment"].nunique())

patient_counts = (
    good_windows_all
    .groupby("patient")
    .size()
    .sort_values(ascending=False)
)

display(patient_counts.head(20))

In [ ]:
# ============================================================
# PATIENT-WISE TRAIN / VALIDATION / TEST SPLIT
# ============================================================

from sklearn.model_selection import train_test_split

# Get unique patients
patients = good_windows_all["patient"].dropna().unique()

print("Total patients:", len(patients))

# ------------------------------------------------------------
# 70% TRAIN, 30% TEMP
# ------------------------------------------------------------

train_patients, temp_patients = train_test_split(
    patients,
    test_size=0.30,
    random_state=42
)

# ------------------------------------------------------------
# 15% VALIDATION, 15% TEST
# ------------------------------------------------------------

val_patients, test_patients = train_test_split(
    temp_patients,
    test_size=0.50,
    random_state=42
)

print("\nPatient split:")
print("Train      :", len(train_patients))
print("Validation :", len(val_patients))
print("Test       :", len(test_patients))

# ------------------------------------------------------------
# CREATE DATAFRAMES
# ------------------------------------------------------------

train_df = good_windows_all[
    good_windows_all["patient"].isin(train_patients)
].copy()

val_df = good_windows_all[
    good_windows_all["patient"].isin(val_patients)
].copy()

test_df = good_windows_all[
    good_windows_all["patient"].isin(test_patients)
].copy()

print("\nWindow split:")
print("Train windows      :", len(train_df))
print("Validation windows :", len(val_df))
print("Test windows       :", len(test_df))

print("\nSegments:")
print("Train segments      :", train_df["segment"].nunique())
print("Validation segments :", val_df["segment"].nunique())
print("Test segments       :", test_df["segment"].nunique())

In [ ]:
# ============================================================
# CHECK FOR PATIENT LEAKAGE
# ============================================================

train_set = set(train_patients)
val_set = set(val_patients)
test_set = set(test_patients)

print("Train ∩ Validation :", train_set.intersection(val_set))
print("Train ∩ Test       :", train_set.intersection(test_set))
print("Validation ∩ Test  :", val_set.intersection(test_set))

In [ ]:
# ============================================================
# BALANCE WINDOWS PER SEGMENT
# ============================================================

MAX_WINDOWS_PER_SEGMENT = 20

def balance_segments(df, max_windows=20, random_state=42):
    
    return (
        df.groupby(
            ["patient", "record", "segment"],
            group_keys=False
        )
        .apply(
            lambda x: x.sample(
                n=min(len(x), max_windows),
                random_state=random_state
            )
        )
        .reset_index(drop=True)
    )


train_balanced = balance_segments(
    train_df,
    MAX_WINDOWS_PER_SEGMENT
)

val_balanced = balance_segments(
    val_df,
    MAX_WINDOWS_PER_SEGMENT
)

test_balanced = balance_segments(
    test_df,
    MAX_WINDOWS_PER_SEGMENT
)

print("=" * 60)
print("BALANCED DATASET")
print("=" * 60)

print("Train      :", len(train_balanced))
print("Validation :", len(val_balanced))
print("Test       :", len(test_balanced))

print("\nPatients:")
print("Train      :", train_balanced["patient"].nunique())
print("Validation :", val_balanced["patient"].nunique())
print("Test       :", test_balanced["patient"].nunique())

print("\nSegments:")
print("Train      :", train_balanced["segment"].nunique())
print("Validation :", val_balanced["segment"].nunique())
print("Test       :", test_balanced["segment"].nunique())

In [ ]:
# ============================================================
# CHECK SBP / DBP DISTRIBUTION
# ============================================================

print("TRAIN SBP")
display(train_balanced["SBP_label"].describe())

print("TRAIN DBP")
display(train_balanced["DBP_label"].describe())

print("\nVALIDATION SBP")
display(val_balanced["SBP_label"].describe())

print("TEST SBP")
display(test_balanced["SBP_label"].describe())

In [ ]:
# ============================================================
# CREATE FINAL DATASET STRUCTURE
# ============================================================

from pathlib import Path

DATASET_ROOT = Path("BP_Dataset")

TRAIN_IMG = DATASET_ROOT / "train" / "images"
VAL_IMG   = DATASET_ROOT / "val" / "images"
TEST_IMG  = DATASET_ROOT / "test" / "images"

TRAIN_IMG.mkdir(parents=True, exist_ok=True)
VAL_IMG.mkdir(parents=True, exist_ok=True)
TEST_IMG.mkdir(parents=True, exist_ok=True)

print("Dataset folders created:")
print(TRAIN_IMG)
print(VAL_IMG)
print(TEST_IMG)

In [ ]:
# ============================================================
# SAVE LABEL FILES
# ============================================================

train_balanced.to_csv(
    DATASET_ROOT / "train" / "labels.csv",
    index=False
)

val_balanced.to_csv(
    DATASET_ROOT / "val" / "labels.csv",
    index=False
)

test_balanced.to_csv(
    DATASET_ROOT / "test" / "labels.csv",
    index=False
)

print("Labels saved successfully.")

print("\nTrain labels:", len(train_balanced))
print("Val labels  :", len(val_balanced))
print("Test labels :", len(test_balanced))

In [ ]:
# ============================================================
# TEST ONE SCALOGRAM
# ============================================================

import matplotlib.pyplot as plt
import pywt
import cv2
import numpy as np
import wfdb

sample = train_balanced.iloc[0]

patient = sample["patient"]
record = sample["record"]
segment = sample["segment"]
window_id = int(sample["window"])

print("Patient :", patient)
print("Record  :", record)
print("Segment :", segment)
print("Window  :", window_id)
print("SBP     :", sample["SBP_label"])
print("DBP     :", sample["DBP_label"])

# ------------------------------------------------------------
# FIND RECORD
# ------------------------------------------------------------

record_path = (
    WAVE_ROOT /
    "waves" /
    patient[:4] /
    patient /
    str(record) /
    str(segment)
)

print("\nRecord path:")
print(record_path)
print("Exists:", record_path.with_suffix(".hea").exists())

# ------------------------------------------------------------
# LOAD RECORD
# ------------------------------------------------------------

rec = wfdb.rdrecord(str(record_path))

channels = [c.upper() for c in rec.sig_name]

print("\nChannels:", channels)
print("Sampling frequency:", rec.fs)

# ------------------------------------------------------------
# FIND PPG
# ------------------------------------------------------------

if "PLETH" in channels:
    ppg_index = channels.index("PLETH")
elif "PPG" in channels:
    ppg_index = channels.index("PPG")
else:
    raise ValueError("No PLETH/PPG channel found.")

ppg = rec.p_signal[:, ppg_index]

# ------------------------------------------------------------
# PREPROCESS
# ------------------------------------------------------------

processed = preprocess_ppg(ppg, rec.fs)

# ------------------------------------------------------------
# EXTRACT THE SELECTED 10-SECOND WINDOW
# ------------------------------------------------------------

WINDOW = int(10 * rec.fs)

start = window_id * WINDOW
end = start + WINDOW

window = processed[start:end]

print("\nWindow samples:", len(window))
print("Expected:", WINDOW)

if len(window) != WINDOW:
    raise ValueError("Incorrect window length.")

# ------------------------------------------------------------
# CWT
# ------------------------------------------------------------

scales = np.arange(1, 129)

coef, frequencies = pywt.cwt(
    window,
    scales,
    "morl"
)

img = np.abs(coef)

# Normalize
img -= img.min()

if img.max() > 0:
    img /= img.max()

img = (img * 255).astype(np.uint8)

# Convert to RGB
img = cv2.applyColorMap(
    img,
    cv2.COLORMAP_JET
)

# Resize
img = cv2.resize(
    img,
    (224, 224),
    interpolation=cv2.INTER_AREA
)

# OpenCV BGR → RGB for matplotlib
img_rgb = cv2.cvtColor(
    img,
    cv2.COLOR_BGR2RGB
)

# ------------------------------------------------------------
# DISPLAY
# ------------------------------------------------------------

plt.figure(figsize=(5, 5))

plt.imshow(img_rgb)

plt.title(
    f"Window {window_id} | "
    f"SBP={sample['SBP_label']:.2f} | "
    f"DBP={sample['DBP_label']:.2f}"
)

plt.axis("off")
plt.show()

print("\nScalogram shape:", img.shape)

In [ ]:
# ============================================================
# GENERATE FINAL SCALOGRAM DATASET
# ============================================================

import os
import cv2
import pywt
import wfdb
import numpy as np
from tqdm import tqdm


def generate_scalograms(
    dataframe,
    output_dir,
    split_name
):
    
    os.makedirs(output_dir, exist_ok=True)

    saved = 0
    failed = 0

    print("=" * 60)
    print(f"GENERATING {split_name.upper()} SCALOGRAMS")
    print("=" * 60)

    for _, row in tqdm(
        dataframe.iterrows(),
        total=len(dataframe),
        desc=f"{split_name}"
    ):

        try:

            patient = row["patient"]
            record = row["record"]
            segment = row["segment"]
            window_id = int(row["window"])

            # ------------------------------------------------
            # RECORD PATH
            # ------------------------------------------------

            record_path = (
                WAVE_ROOT /
                "waves" /
                patient[:4] /
                patient /
                str(record) /
                str(segment)
            )

            # ------------------------------------------------
            # LOAD RECORD
            # ------------------------------------------------

            rec = wfdb.rdrecord(
                str(record_path)
            )

            channels = [
                c.upper()
                for c in rec.sig_name
            ]

            # ------------------------------------------------
            # FIND PPG
            # ------------------------------------------------

            if "PLETH" in channels:

                ppg_index = channels.index(
                    "PLETH"
                )

            elif "PPG" in channels:

                ppg_index = channels.index(
                    "PPG"
                )

            else:

                failed += 1
                continue

            ppg = rec.p_signal[
                :,
                ppg_index
            ]

            # ------------------------------------------------
            # PREPROCESS
            # ------------------------------------------------

            processed = preprocess_ppg(
                ppg,
                rec.fs
            )

            # ------------------------------------------------
            # EXTRACT WINDOW
            # ------------------------------------------------

            WINDOW = int(
                WINDOW_SECONDS * rec.fs
            )

            start = window_id * WINDOW
            end = start + WINDOW

            window = processed[
                start:end
            ]

            if len(window) != WINDOW:

                failed += 1
                continue

            # ------------------------------------------------
            # CWT
            # ------------------------------------------------

            scales = np.arange(
                1,
                129
            )

            coef, _ = pywt.cwt(
                window,
                scales,
                "morl"
            )

            img = np.abs(coef)

            # ------------------------------------------------
            # NORMALIZE
            # ------------------------------------------------

            img -= img.min()

            if img.max() > 0:

                img /= img.max()

            img = (
                img * 255
            ).astype(
                np.uint8
            )

            # ------------------------------------------------
            # COLORMAP
            # ------------------------------------------------

            img = cv2.applyColorMap(
                img,
                cv2.COLORMAP_JET
            )

            # ------------------------------------------------
            # RESIZE
            # ------------------------------------------------

            img = cv2.resize(
                img,
                (224, 224),
                interpolation=cv2.INTER_AREA
            )

            # ------------------------------------------------
            # SAVE
            # ------------------------------------------------

            filename = (
                f"{patient}_"
                f"{record}_"
                f"{segment}_"
                f"{window_id}.png"
            )

            save_path = (
                os.path.join(
                    output_dir,
                    filename
                )
            )

            success = cv2.imwrite(
                save_path,
                img
            )

            if success:

                saved += 1

            else:

                failed += 1

        except Exception as e:

            failed += 1

    print("\n" + "=" * 60)
    print(f"{split_name.upper()} COMPLETE")
    print("=" * 60)

    print("Expected :", len(dataframe))
    print("Saved    :", saved)
    print("Failed   :", failed)

    return saved

In [ ]:
# ============================================================
# GENERATE TRAIN / VAL / TEST
# ============================================================

train_saved = generate_scalograms(
    train_balanced,
    TRAIN_IMG,
    "train"
)

val_saved = generate_scalograms(
    val_balanced,
    VAL_IMG,
    "validation"
)

test_saved = generate_scalograms(
    test_balanced,
    TEST_IMG,
    "test"
)

print("\n" + "=" * 60)
print("FINAL SCALOGRAM SUMMARY")
print("=" * 60)

print("Train :", train_saved)
print("Val   :", val_saved)
print("Test  :", test_saved)
print("Total :", train_saved + val_saved + test_saved)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
import sys
print(sys.executable)